# Inteligencia de Negocios
## Práctica Semana 2 – ETL: Limpieza y Transformación de Datos
## Grupo # 3

### ALEJANDRA BEATRIZ TELLO GONZÁLEZ
### PABLO RAMIRO VALLEJO ZÚÑIGA

---

### Descripción del proyecto

El proyecto trabaja con datos del dominio de **Ciberseguridad Bancaria**. Las tres fuentes de datos provienen de:

| Fuente | Tipo | Descripción |
|--------|------|-------------|
| `api_security_events` | PostgreSQL | Eventos de seguridad en canales bancarios digitales |
| `incidentes_seguridad.csv` | CSV | Incidentes de red detectados por el IDS/SOC |
| `vulnerabilidades_cve.json` | JSON | Vulnerabilidades CVE identificadas en escaneos |

En esta práctica se aplica la etapa **Transform** del proceso ETL: limpieza, normalización, selección de columnas, transformaciones y generación de identificadores, preparando los datos para su posterior carga en un Data Warehouse.

---
## Importación de librerías

Se importan las librerías necesarias para la extracción, transformación y validación de los datos.

In [1]:
import pandas as pd
import numpy as np
import json
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
print('✅ Librerías cargadas correctamente')

✅ Librerías cargadas correctamente


---
## 2. Configuración segura del proyecto

Las credenciales de conexión a PostgreSQL se gestionan mediante un archivo `.env`, nunca escritas directamente en el código fuente.

**Variables de entorno utilizadas:**
- `POSTGRES_USER` – usuario de la base de datos
- `POSTGRES_PASSWORD` – contraseña de acceso
- `POSTGRES_DB` – nombre de la base de datos
- `POSTGRES_HOST` – host del contenedor Docker
- `POSTGRES_PORT` – puerto de conexión

**¿Por qué no escribir credenciales en el código?**  
En entornos bancarios, el código fuente se versiona en repositorios Git que pueden ser accedidos por múltiples personas o incluso ser públicos. Exponer credenciales directamente implicaría un riesgo crítico de seguridad: acceso no autorizado a bases de datos de producción, violación de normativas como PCI-DSS e ISO 27001, y potencial exfiltración de datos sensibles.

**Aporte a la seguridad del proyecto:**  
El uso de `.env` con `python-dotenv` garantiza que las credenciales se mantengan fuera del repositorio (el `.env` se agrega al `.gitignore`), siguiendo el principio de separación de configuración y código.

In [2]:
DB_USER     = os.getenv('POSTGRES_USER')
DB_PASSWORD = os.getenv('POSTGRES_PASSWORD')
DB_NAME     = os.getenv('POSTGRES_DB')
DB_HOST     = os.getenv('POSTGRES_HOST')
DB_PORT     = os.getenv('POSTGRES_PORT', '5432')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)
print(f'🔌 Conectando a: {DB_HOST}:{DB_PORT}/{DB_NAME} como usuario \'\'{DB_USER}\'\'\n')
print('Variables de entorno cargadas: POSTGRES_USER, POSTGRES_PASSWORD, POSTGRES_DB, POSTGRES_HOST, POSTGRES_PORT')

🔌 Conectando a: None:5432/None como usuario ''None''

Variables de entorno cargadas: POSTGRES_USER, POSTGRES_PASSWORD, POSTGRES_DB, POSTGRES_HOST, POSTGRES_PORT


---
## 1. Exploración inicial de las fuentes de datos

Antes de aplicar cualquier transformación, se realiza una exploración completa de cada fuente para identificar problemas de calidad de datos.

In [3]:
# ── Extracción de las tres fuentes ─────────────────────────────────────────
df_pg   = pd.read_sql('SELECT * FROM api_security_events', engine)
df_csv  = pd.read_csv('data/incidentes_seguridad.csv')
with open('data/vulnerabilidades_cve.json', 'r', encoding='utf-8') as f:
    df_json = pd.DataFrame(json.load(f))

print('Fuentes cargadas:')
print(f'  df_pg   : {df_pg.shape[0]} filas × {df_pg.shape[1]} columnas')
print(f'  df_csv  : {df_csv.shape[0]} filas × {df_csv.shape[1]} columnas')
print(f'  df_json : {df_json.shape[0]} filas × {df_json.shape[1]} columnas')

OperationalError: (psycopg2.OperationalError) could not translate host name "None" to address: Name or service not known

(Background on this error at: https://sqlalche.me/e/20/e3q8)

### 1.1 Exploración – API Security Events (PostgreSQL)

**Hallazgos identificados:**
- La columna `user_agent` contiene valores nulos (~10%) correspondientes a tráfico automatizado sin cabecera HTTP
- `event_date` viene como tipo `object` (string), debe convertirse a `datetime`
- `risk_score` es numérico entero del 1 al 5, puede categorizarse para análisis
- `channel` y `event_type` tienen textos que deben normalizarse a mayúsculas consistentes
- Columna de relación con otras fuentes: `event_type` ↔ `attack_type` (CSV) y `severity` ↔ `cvss_severity` (JSON)

In [ ]:
print('── Tipos de datos ──')
print(df_pg.dtypes)
print(f'\n── Valores nulos por columna ──')
print(df_pg.isnull().sum())
print(f'\n── Registros duplicados: {df_pg.duplicated().sum()} ──')
print(f'\n── Valores únicos en channel ──')
print(df_pg['channel'].unique())
print(f'\n── Valores únicos en event_type ──')
print(df_pg['event_type'].unique())

### 1.2 Exploración – Incidentes de Seguridad (CSV)

**Hallazgos identificados:**
- `analyst_assigned` tiene nulos (~10%) donde el incidente fue gestionado automáticamente por el SIEM
- `timestamp` es string, debe convertirse a `datetime`
- `false_positive` es booleano, útil para filtrar en la capa de transformación
- `severity` tiene valores mixtos que deben estandarizarse (Low/Medium/High/Critical)
- Posible relación con df_json a través de `severity` ↔ `cvss_severity`

In [ ]:
print('── Tipos de datos ──')
print(df_csv.dtypes)
print(f'\n── Valores nulos por columna ──')
print(df_csv.isnull().sum())
print(f'\n── Registros duplicados: {df_csv.duplicated().sum()} ──')
print(f'\n── Distribución de severity ──')
print(df_csv['severity'].value_counts())
print(f'\n── Distribución de attack_type ──')
print(df_csv['attack_type'].value_counts())

### 1.3 Exploración – Vulnerabilidades CVE (JSON)

**Hallazgos identificados:**
- `patch_applied` tiene nulos (~30%) para vulnerabilidades sin diagnóstico de parche — hallazgo crítico en auditorías PCI-DSS
- `discovery_date` es string, debe convertirse a `datetime`
- `cvss_score` es float, puede usarse para generar categorías de riesgo adicionales
- `exploited_in_wild` es booleano relevante para priorización de remediación
- Columna de relación: `cvss_severity` ↔ `severity` (CSV) y `affected_hosts` como métrica agregable

In [ ]:
print('── Tipos de datos ──')
print(df_json.dtypes)
print(f'\n── Valores nulos por columna ──')
print(df_json.isnull().sum())
print(f'\n── Registros duplicados: {df_json.duplicated().sum()} ──')
print(f'\n── Distribución de cvss_severity ──')
print(df_json['cvss_severity'].value_counts())
print(f'\n── Rango de cvss_score ──')
print(df_json['cvss_score'].describe())

---
## 3. Limpieza de datos

Se definen funciones reutilizables de limpieza para cada fuente. El diseño modular permite aplicarlas en pipelines ETL futuros sin modificar el código principal.

In [ ]:
# ── Limpieza: API Security Events (PostgreSQL) ─────────────────────────────
def limpiar_api_events(df):
    """Limpia y normaliza el DataFrame de eventos de seguridad en APIs bancarias."""
    df = df.copy()
    # 1. Convertir fecha a datetime
    df['event_date'] = pd.to_datetime(df['event_date'])
    # 2. Rellenar nulos en user_agent con valor descriptivo
    df['user_agent'] = df['user_agent'].fillna('Unknown/Automated')
    # 3. Normalizar textos a Title Case para consistencia
    df['channel']    = df['channel'].str.strip().str.title()
    df['event_type'] = df['event_type'].str.strip().str.title()
    df['http_method']= df['http_method'].str.strip().str.upper()
    df['country_code']= df['country_code'].str.strip().str.upper()
    # 4. Eliminar duplicados
    df = df.drop_duplicates()
    # 5. Convertir resolved a booleano explícito
    df['resolved'] = df['resolved'].astype(bool)
    return df

df_pg_clean = limpiar_api_events(df_pg)
print(f'✅ df_pg limpio: {df_pg_clean.shape[0]} registros')
print(f'   Nulos restantes: {df_pg_clean.isnull().sum().sum()}')

In [ ]:
# ── Limpieza: Incidentes de Seguridad (CSV) ────────────────────────────────
def limpiar_incidentes(df):
    """Limpia y normaliza el DataFrame de incidentes de red del SOC."""
    df = df.copy()
    # 1. Convertir timestamp a datetime
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    # 2. Rellenar analista no asignado
    df['analyst_assigned'] = df['analyst_assigned'].fillna('Sistema Automatico')
    # 3. Estandarizar textos
    df['attack_type']     = df['attack_type'].str.strip().str.title()
    df['severity']        = df['severity'].str.strip().str.title()
    df['protocol']        = df['protocol'].str.strip().str.upper()
    df['status']          = df['status'].str.strip().str.title()
    df['affected_system'] = df['affected_system'].str.strip().str.title()
    # 4. Eliminar duplicados
    df = df.drop_duplicates()
    # 5. Convertir false_positive a booleano
    df['false_positive'] = df['false_positive'].astype(bool)
    return df

df_csv_clean = limpiar_incidentes(df_csv)
print(f'✅ df_csv limpio: {df_csv_clean.shape[0]} registros')
print(f'   Nulos restantes: {df_csv_clean.isnull().sum().sum()}')

In [ ]:
# ── Limpieza: Vulnerabilidades CVE (JSON) ──────────────────────────────────
def limpiar_vulnerabilidades(df):
    """Limpia y normaliza el DataFrame de vulnerabilidades CVE."""
    df = df.copy()
    # 1. Convertir discovery_date a datetime
    df['discovery_date'] = pd.to_datetime(df['discovery_date'])
    # 2. Tratar nulos en patch_applied: None indica estado desconocido
    df['patch_applied'] = df['patch_applied'].fillna(False).astype(bool)
    # 3. Estandarizar textos
    df['cvss_severity']      = df['cvss_severity'].str.strip().str.title()
    df['vulnerability_type'] = df['vulnerability_type'].str.strip().str.title()
    df['system']             = df['system'].str.strip()
    df['vendor']             = df['vendor'].str.strip()
    # 4. Eliminar duplicados
    df = df.drop_duplicates()
    # 5. Redondear cvss_score a 1 decimal para consistencia
    df['cvss_score'] = df['cvss_score'].round(1)
    return df

df_json_clean = limpiar_vulnerabilidades(df_json)
print(f'✅ df_json limpio: {df_json_clean.shape[0]} registros')
print(f'   Nulos restantes: {df_json_clean.isnull().sum().sum()}')

---
## 4. Selección de columnas relevantes

Se evalúa qué columnas aportan valor analítico al proyecto y cuáles se descartan para optimizar el modelo de datos.

### Criterios de selección
- **Se mantienen:** columnas con valor analítico directo para el DW (métricas, dimensiones, claves)
- **Se eliminan:** columnas técnicas de bajo valor analítico o con alta cardinalidad sin estructura

### Columnas de relación entre fuentes
| Columna fuente A | DataFrame A | Columna fuente B | DataFrame B |
|-----------------|-------------|-----------------|-------------|
| `severity` | df_csv | `cvss_severity` | df_json |
| `event_type` | df_pg | `attack_type` | df_csv |
| `affected_system` | df_csv | `system` | df_json |

In [ ]:
# ── Selección de columnas: API Security Events ─────────────────────────────
# Se elimina: source_ip (alta cardinalidad, no agrupable), user_agent (reemplazado por flag)
cols_pg = ['event_id','event_date','channel','event_type','http_method',
           'endpoint','response_code','response_time_ms','risk_score',
           'resolved','country_code']
df_pg_sel = df_pg_clean[cols_pg].copy()
print('df_pg columnas seleccionadas:', df_pg_sel.shape[1])
print('Eliminadas: user_agent (reemplazada por Unknown/Automated, bajo valor dimensional)')

# ── Selección de columnas: Incidentes CSV ─────────────────────────────────
# Se elimina: source_ip (alta cardinalidad), incident_id se renombra a id_incidente
cols_csv = ['incident_id','timestamp','attack_type','protocol','destination_port',
            'packets_sent','bytes_transferred','duration_seconds','severity',
            'status','affected_system','analyst_assigned','false_positive']
df_csv_sel = df_csv_clean[cols_csv].copy()
print('\ndf_csv columnas seleccionadas:', df_csv_sel.shape[1])
print('Eliminadas: source_ip (PII/alta cardinalidad, no agrupable para DW)')

# ── Selección de columnas: Vulnerabilidades JSON ───────────────────────────
# Se mantienen todas: cada columna aporta dimensión o métrica relevante
cols_json = ['vuln_id','cve_id','system','vendor','vulnerability_type','cvss_score',
             'cvss_severity','patch_available','patch_applied','discovery_date',
             'remediation_deadline_days','affected_hosts','exploited_in_wild']
df_json_sel = df_json_clean[cols_json].copy()
print('\ndf_json columnas seleccionadas:', df_json_sel.shape[1])
print('Eliminadas: ninguna — todas las columnas tienen valor analítico o dimensional')

---
## 5. Transformaciones requeridas

Se aplican transformaciones que enriquecen los datos y preparan columnas derivadas para el análisis en el Data Warehouse.

In [ ]:
# ── Transformaciones: API Security Events ──────────────────────────────────

# Nueva columna: categoría de riesgo a partir de risk_score (1-5)
df_pg_sel['risk_category'] = df_pg_sel['risk_score'].apply(
    lambda x: 'Crítico' if x == 5 else 'Alto' if x == 4 else 'Medio' if x == 3 else 'Bajo'
)

# Nueva columna: flag de respuesta lenta (>2000ms puede indicar ataque en curso)
df_pg_sel['respuesta_lenta'] = df_pg_sel['response_time_ms'].apply(
    lambda x: True if x > 2000 else False
)

# Nueva columna: mes del evento para agrupación temporal en DW
df_pg_sel['mes_evento'] = df_pg_sel['event_date'].dt.to_period('M').astype(str)

# Nueva columna: flag de código de error HTTP (>=400)
df_pg_sel['es_error_http'] = df_pg_sel['response_code'].apply(
    lambda x: True if x >= 400 else False
)

print('✅ Transformaciones aplicadas a df_pg:')
print('   + risk_category: categorización del risk_score')
print('   + respuesta_lenta: flag de latencia anómala >2000ms')
print('   + mes_evento: período mensual para agrupación temporal')
print('   + es_error_http: flag de respuestas HTTP de error')
df_pg_sel[['event_id','risk_score','risk_category','response_time_ms','respuesta_lenta','mes_evento']].head()

In [ ]:
# ── Transformaciones: Incidentes CSV ──────────────────────────────────────

# Nueva columna: nivel numérico de severidad para ordenación y scoring
severity_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
df_csv_sel['severity_nivel'] = df_csv_sel['severity'].map(severity_map)

# Nueva columna: duración del incidente en minutos (más legible que segundos)
df_csv_sel['duration_minutes'] = (df_csv_sel['duration_seconds'] / 60).round(2)

# Nueva columna: volumen de tráfico categorizado
df_csv_sel['volumen_trafico'] = df_csv_sel['bytes_transferred'].apply(
    lambda x: 'Alto' if x > 5_000_000 else 'Medio' if x > 500_000 else 'Bajo'
)

# Nueva columna: mes del incidente para alineación temporal con df_pg
df_csv_sel['mes_incidente'] = df_csv_sel['timestamp'].dt.to_period('M').astype(str)

# Filtrar falsos positivos confirmados (no incluir en DW de incidentes reales)
df_csv_real = df_csv_sel[df_csv_sel['false_positive'] == False].copy()

print('✅ Transformaciones aplicadas a df_csv:')
print('   + severity_nivel: escala numérica 1-4')
print('   + duration_minutes: duración en minutos')
print('   + volumen_trafico: categorización de bytes')
print('   + mes_incidente: período mensual')
print(f'   Registros reales (sin falsos positivos): {len(df_csv_real)} de {len(df_csv_sel)}')
df_csv_real[['incident_id','attack_type','severity','severity_nivel','duration_minutes','volumen_trafico']].head()

In [ ]:
# ── Transformaciones: Vulnerabilidades JSON ────────────────────────────────

# Nueva columna: prioridad de remediación basada en CVSS + explotación activa
def calcular_prioridad(row):
    if row['exploited_in_wild'] and row['cvss_score'] >= 7:
        return 'Inmediata'
    elif row['cvss_score'] >= 9:
        return 'Urgente'
    elif row['cvss_score'] >= 7:
        return 'Alta'
    elif row['cvss_score'] >= 4:
        return 'Media'
    else:
        return 'Baja'

df_json_sel['prioridad_remediacion'] = df_json_sel.apply(calcular_prioridad, axis=1)

# Nueva columna: días transcurridos desde el descubrimiento
df_json_sel['dias_desde_descubrimiento'] = (
    pd.Timestamp('today') - df_json_sel['discovery_date']
).dt.days

# Nueva columna: estado consolidado del parche
df_json_sel['estado_parche'] = df_json_sel.apply(
    lambda row: 'Aplicado' if row['patch_applied']
    else ('Disponible' if row['patch_available'] else 'Sin parche'),
    axis=1
)

# Normalizar cvss_severity para alinearlo con severity del CSV
df_json_sel['severity'] = df_json_sel['cvss_severity'].str.title()

print('✅ Transformaciones aplicadas a df_json:')
print('   + prioridad_remediacion: basada en CVSS + explotación activa')
print('   + dias_desde_descubrimiento: antigüedad del hallazgo')
print('   + estado_parche: consolidación de patch_available + patch_applied')
print('   + severity: normalizada para join con df_csv')
df_json_sel[['vuln_id','cve_id','cvss_score','prioridad_remediacion','estado_parche','dias_desde_descubrimiento']].head()

---
## 6. Generación de identificadores

Se generan identificadores únicos y secuenciales para facilitar las relaciones entre tablas en el Data Warehouse y las operaciones de carga incremental en el ETL.

**Identificadores creados:**
- `etl_id_evento`: ID secuencial para df_pg — permite upserts en la tabla de hechos de eventos
- `etl_id_incidente`: ID secuencial para df_csv — clave subrogada en la tabla de hechos de incidentes
- `etl_id_vuln`: ID secuencial para df_json — clave subrogada en la dimensión de vulnerabilidades

Los IDs secuenciales son independientes de las claves naturales (`event_id`, `incident_id`, `vuln_id`), lo que permite manejar inserciones de distintas fuentes sin colisión de claves.

In [ ]:
# ── Generación de identificadores secuenciales para el DW ─────────────────

# Resetear índice y crear ID ETL secuencial (base 1)
df_pg_sel   = df_pg_sel.reset_index(drop=True)
df_csv_real = df_csv_real.reset_index(drop=True)
df_json_sel = df_json_sel.reset_index(drop=True)

df_pg_sel['etl_id_evento']     = df_pg_sel.index + 1
df_csv_real['etl_id_incidente'] = df_csv_real.index + 1
df_json_sel['etl_id_vuln']     = df_json_sel.index + 1

# Mover el ID ETL al frente del DataFrame
for df, col in [(df_pg_sel,'etl_id_evento'),(df_csv_real,'etl_id_incidente'),(df_json_sel,'etl_id_vuln')]:
    cols = [col] + [c for c in df.columns if c != col]
    df   = df[cols]

print('Identificadores ETL generados:')
print(f'  etl_id_evento     : 1 a {len(df_pg_sel)}    → tabla FACT_EVENTOS')
print(f'  etl_id_incidente  : 1 a {len(df_csv_real)}  → tabla FACT_INCIDENTES')
print(f'  etl_id_vuln       : 1 a {len(df_json_sel)}   → tabla DIM_VULNERABILIDADES')
print()
print('Uso en DW: los ETL IDs permiten joins entre tablas de hechos y dimensiones')
print('sin depender de claves naturales que pueden cambiar entre cargas.')

---
## 7. Validación de resultados

Se verifica que los DataFrames transformados sean consistentes y estén listos para la carga en el Data Warehouse.

In [ ]:
# ── Validación: API Security Events ───────────────────────────────────────
print('══ VALIDACIÓN df_pg_sel ══')
print(f'  Shape        : {df_pg_sel.shape}')
print(f'  Nulos totales: {df_pg_sel.isnull().sum().sum()}')
print(f'  Duplicados   : {df_pg_sel.duplicated().sum()}')
print('  Tipos de datos:')
print(df_pg_sel.dtypes)
print('\nVista previa:')
df_pg_sel.head(3)

In [ ]:
# ── Validación: Incidentes CSV ─────────────────────────────────────────────
print('══ VALIDACIÓN df_csv_real ══')
print(f'  Shape        : {df_csv_real.shape}')
print(f'  Nulos totales: {df_csv_real.isnull().sum().sum()}')
print(f'  Duplicados   : {df_csv_real.duplicated().sum()}')
print('  Distribución severity_nivel:')
print(df_csv_real['severity_nivel'].value_counts().sort_index())
print('\nVista previa:')
df_csv_real.head(3)

In [ ]:
# ── Validación: Vulnerabilidades JSON ─────────────────────────────────────
print('══ VALIDACIÓN df_json_sel ══')
print(f'  Shape        : {df_json_sel.shape}')
print(f'  Nulos totales: {df_json_sel.isnull().sum().sum()}')
print(f'  Duplicados   : {df_json_sel.duplicated().sum()}')
print('  Distribución prioridad_remediacion:')
print(df_json_sel['prioridad_remediacion'].value_counts())
print('  Distribución estado_parche:')
print(df_json_sel['estado_parche'].value_counts())
print('\nVista previa:')
df_json_sel.head(3)

In [ ]:
# ── Resumen final de transformación ───────────────────────────────────────
resumen = pd.DataFrame({
    'DataFrame'       : ['df_pg_sel', 'df_csv_real', 'df_json_sel'],
    'Filas originales': [df_pg.shape[0], df_csv.shape[0], df_json.shape[0]],
    'Filas limpias'   : [df_pg_sel.shape[0], df_csv_real.shape[0], df_json_sel.shape[0]],
    'Columnas orig'   : [df_pg.shape[1], df_csv.shape[1], df_json.shape[1]],
    'Columnas final'  : [df_pg_sel.shape[1], df_csv_real.shape[1], df_json_sel.shape[1]],
    'Nulos restantes' : [df_pg_sel.isnull().sum().sum(),
                         df_csv_real.isnull().sum().sum(),
                         df_json_sel.isnull().sum().sum()]
})
print('══ RESUMEN ETL – FASE TRANSFORM ══')
print(resumen.to_string(index=False))

---
## 8. Correlación de las tres fuentes de datos

Para integrar las tres fuentes en un DataFrame unificado, se estandariza la columna de severidad como clave común:

| Fuente | Columna original | Columna estandarizada |
|--------|-----------------|----------------------|
| PostgreSQL |  (1-5) |  (Low/Medium/High/Critical) |
| CSV |  (Low/Medium/High/Critical) |  |
| JSON |  (Low/Medium/High/Critical) |  |

El DataFrame correlacionado permite analizar, para cada nivel de severidad:
- Cuántos **eventos de API** ocurrieron (PostgreSQL)
- Cuántos **incidentes de red** se detectaron (CSV)
- Cuántas **vulnerabilidades CVE** existen (JSON)
- Métricas agregadas: tiempo de respuesta promedio, bytes transferidos, score CVSS promedio

In [ ]:
# ── Estandarización de severity en las 3 fuentes ──────────────────────────

# PostgreSQL: mapear risk_score (1-5) a niveles de severidad
risk_to_severity = {1: "Low", 2: "Low", 3: "Medium", 4: "High", 5: "Critical"}
df_pg_sel["severity_std"] = df_pg_sel["risk_score"].map(risk_to_severity)

# CSV: severity ya está en el formato correcto, solo renombrar para alinear
df_csv_real["severity_std"] = df_csv_real["severity"].str.title()

# JSON: cvss_severity ya está estandarizado en la limpieza
df_json_sel["severity_std"] = df_json_sel["cvss_severity"].str.title()

print("Valores únicos de severity_std por fuente:")
print("  PostgreSQL:", sorted(df_pg_sel["severity_std"].unique()))
print("  CSV       :", sorted(df_csv_real["severity_std"].unique()))
print("  JSON      :", sorted(df_json_sel["severity_std"].unique()))

### Agregación por severity_std

Se agregan métricas de cada fuente agrupadas por  para generar la vista correlacionada.
Este DataFrame unificado es la base para el modelo dimensional del Data Warehouse.

In [ ]:
# ── Agregación por severity_std en cada fuente ────────────────────────────

# Métricas desde PostgreSQL: conteo de eventos y tiempo de respuesta promedio
agg_pg = (
    df_pg_sel.groupby("severity_std")
    .agg(
        total_eventos=("event_id", "count"),
        resp_time_promedio_ms=("response_time_ms", "mean"),
        eventos_no_resueltos=("resolved", lambda x: (~x).sum())
    )
    .reset_index()
    .rename(columns={"resp_time_promedio_ms": lambda c: c})
)
agg_pg["resp_time_promedio_ms"] = agg_pg["resp_time_promedio_ms"].round(1)

# Métricas desde CSV: conteo de incidentes y bytes promedio
agg_csv = (
    df_csv_real.groupby("severity_std")
    .agg(
        total_incidentes=("incident_id", "count"),
        bytes_promedio=("bytes_transferred", "mean"),
        duracion_promedio_min=("duration_minutes", "mean")
    )
    .reset_index()
)
agg_csv["bytes_promedio"]        = agg_csv["bytes_promedio"].round(0).astype(int)
agg_csv["duracion_promedio_min"]  = agg_csv["duracion_promedio_min"].round(2)

# Métricas desde JSON: conteo de CVEs, score CVSS promedio y hosts afectados
agg_json = (
    df_json_sel.groupby("severity_std")
    .agg(
        total_vulnerabilidades=("vuln_id", "count"),
        cvss_score_promedio=("cvss_score", "mean"),
        hosts_afectados_total=("affected_hosts", "sum"),
        pendientes_parche=("patch_applied", lambda x: (~x).sum())
    )
    .reset_index()
)
agg_json["cvss_score_promedio"] = agg_json["cvss_score_promedio"].round(2)

print("Agregaciones generadas:")
print(f"  agg_pg   : {agg_pg.shape}")
print(f"  agg_csv  : {agg_csv.shape}")
print(f"  agg_json : {agg_json.shape}")

### DataFrame correlacionado final

Se realiza el  de las tres agregaciones usando  como clave común.
El resultado es un DataFrame unificado que cruza información de las tres fuentes
por nivel de severidad, listo para ser cargado en el Data Warehouse.

In [ ]:
# ── Merge de las 3 fuentes por severity_std ───────────────────────────────

df_correlacionado = (
    agg_pg
    .merge(agg_csv,  on="severity_std", how="outer")
    .merge(agg_json, on="severity_std", how="outer")
)

# Ordenar por severidad de menor a mayor
orden_severidad = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}
df_correlacionado["orden"] = df_correlacionado["severity_std"].map(orden_severidad)
df_correlacionado = df_correlacionado.sort_values("orden").drop(columns="orden").reset_index(drop=True)

# Rellenar nulos en caso de niveles sin datos en alguna fuente
df_correlacionado = df_correlacionado.fillna(0)

print("══ DATAFRAME CORRELACIONADO – Vista unificada por severidad ══")
print(f"Shape: {df_correlacionado.shape}")
print()
df_correlacionado

### Interpretación del DataFrame correlacionado

El DataFrame unificado permite responder preguntas de negocio clave para el área de seguridad:

- **¿Qué nivel de severidad concentra más incidentes y eventos simultáneamente?** → indica el frente de ataque más activo
- **¿Los niveles Critical tienen vulnerabilidades sin parche?** →  cuantifica el riesgo residual
- **¿Los eventos de alta severidad generan mayor latencia en las APIs?** → correlación entre  y 
- **¿Qué nivel acumula más hosts afectados?** → priorización de remediación por impacto

In [ ]:
# ── Validación del DataFrame correlacionado ────────────────────────────────
print("Nulos en df_correlacionado:", df_correlacionado.isnull().sum().sum())
print("Duplicados              :", df_correlacionado.duplicated().sum())
print("Tipos de datos:")
print(df_correlacionado.dtypes)
print()
print("── Resumen por nivel de severidad ──")
for _, row in df_correlacionado.iterrows():
    print(f"  {row["severity_std"]:8} | eventos: {int(row["total_eventos"]):3} | "
          f"incidentes: {int(row["total_incidentes"]):3} | "
          f"CVEs: {int(row["total_vulnerabilidades"]):2} | "
          f"sin parche: {int(row["pendientes_parche"]):2} | "
          f"CVSS prom: {row["cvss_score_promedio"]}")

---
# Reflexión Individual Final
## Alejandra Beatriz Tello González

Esta semana comprendí que la transformación de datos no es un paso técnico menor, sino el núcleo del proceso ETL: la calidad del análisis posterior depende completamente de la limpieza y estructuración aplicada aquí.

En mi trabajo como especialista en seguridad de la información en el sector bancario, uno de los problemas más recurrentes es la fragmentación de datos entre sistemas: los logs del WAF, los reportes del SIEM y los resultados de escaneos de vulnerabilidades llegan en formatos distintos, con timestamps inconsistentes, campos nulos sin criterio y severidades escritas de formas diferentes según la herramienta. Aplicar funciones de limpieza como las desarrolladas en esta práctica —estandarización de textos, conversión de fechas, imputación de nulos con valores descriptivos— permitiría consolidar estas fuentes en un pipeline reproducible.

Los datos que manejo incluyen eventos de autenticación fallida en APIs, alertas de intrusión por canal digital y CVEs con estado de parche. Los errores más comunes son: campos de severidad con mayúsculas inconsistentes, IPs duplicadas por reintentos, y fechas en zonas horarias distintas entre sistemas. Funciones lambda para categorizar riesgo y normalizar formatos resolverían estos problemas de forma automatizada.

Documentar cada paso en el notebook es esencial en entornos regulados: una auditoría de PCI-DSS o de la Superintendencia de Bancos puede requerir evidencia de cómo se procesaron los datos. El notebook actúa como trazabilidad auditada del pipeline ETL, justificando cada decisión de transformación con contexto técnico y de negocio.